# DataMate Multi-Stage Code-Generation Agent (Qwen2.5-Coder-7B-Instruct) — Google Colab Pro

Notebook này được cấu hình chuyên biệt cho **Google Colab Pro (GPU A100 / L4 / T4)** để chạy toàn bộ pipeline **DataMate (Zalo AI Challenge 2026)**.

Hỗ trợ **2 chế độ vận hành**:
1. **Chế độ 1 (Chạy End-to-End 100% trên Colab Pro + Google Drive)**: Mount Google Drive cá nhân, đồng bộ thư mục `zalo AI`, kiểm thử `pytest`, nạp `Qwen/Qwen2.5-Coder-7B-Instruct` lên GPU và chạy `run_pipeline.py` + `evaluate.py`.
2. **Chế độ 2 (Colab Pro làm Remote GPU Server cho máy Mac)**: Mở cổng API tương thích OpenAI qua Cloudflare Tunnel để máy Mac tại `/Applications/zalo AI` gọi trực tiếp GPU Colab Pro.

# DataMate — 3 Cell Độc Lập Chạy 1-Click Trên Google Colab Pro
Mỗi Cell dưới đây là một quy trình **khép kín 100%**:
1. Tự động kết nối Google Drive.
2. Tự động `git pull` code mới nhất từ GitHub (`https://github.com/25520969-netizen/Datamate.git`) trong 0.5 giây.
3. Tự động nạp dữ liệu từ `zalo_data.zip` trên Drive (nếu chưa có trên SSD Colab).
4. Tự động khởi động vLLM Server.
5. Chạy pipeline đa luồng và lưu kết quả về Google Drive.
6. Tự động ngắt GPU (nếu bật `AUTO_SHUTDOWN`).

👉 Bạn chỉ cần chọn và bấm **Run ▶️** vào Cell bạn muốn chạy!

## 🟢 Cell 1: Chạy Benchmark 300 câu mẫu (Train Sample — ~4 đến 5 phút)

In [ ]:
# ==============================================================
# CELL 1: BENCHMARK 300 CÂU MẪU
# Benchmark 300 câu mẫu (Train Sample - Thời gian ~4 đến 5 phút)
# ==============================================================
from google.colab import drive
import os, zipfile, subprocess, time, shutil
from pathlib import Path

# CẤU HÌNH
AUTO_SHUTDOWN = False  # True = tự động tắt GPU sau khi xong để tiết kiệm Compute Units
CONCURRENCY = 8        # Số luồng chạy song song trên GPU

# 1. Mount Google Drive
drive.mount('/content/drive')
WORK_DIR = Path('/content/Datamate')

# 2. TỰ ĐỘNG KÉO CODE MỚI NHẤT TỪ GITHUB (Mất 0.5s)
if not WORK_DIR.exists():
    print("🚀 Đang tải code từ GitHub...")
    !git clone https://github.com/25520969-netizen/Datamate.git $WORK_DIR
    %cd $WORK_DIR
else:
    %cd $WORK_DIR
    print("⚡ Đang cập nhật code mới nhất từ GitHub...")
    !git pull

# 3. Nạp Data từ Drive vào SSD Colab (Chỉ giải nén LẦN ĐẦU TIÊN)
DRIVE_ZIP = Path('/content/drive/MyDrive/zalo_data.zip')
if not (WORK_DIR / 'train' / 'data').exists():
    if not DRIVE_ZIP.exists():
        raise FileNotFoundError("Chưa tìm thấy 'MyDrive/zalo_data.zip'. Vui lòng tải file lên Google Drive trước!")
    print("📦 Lần đầu chạy: Đang giải nén bộ dữ liệu 183 MB vào SSD Colab...")
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(WORK_DIR)
    print("✅ Đã nạp xong toàn bộ dữ liệu Train & Public Test!")
else:
    print("⚡ Dữ liệu đã có sẵn trên SSD Colab, không cần giải nén lại!")

# 4. Cài đặt thư viện cần thiết (nếu thiếu)
!pip install -q vllm duckdb openpyxl pyarrow pytest

# 5. Khởi động vLLM Server (nếu chưa chạy)
if os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
    print("⏳ Đang khởi động vLLM Server trên GPU Colab (chế độ siêu tốc ~45s)...")
    os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")
    subprocess.Popen([
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
        "--port", "8000",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--enable-prefix-caching",
        "--enforce-eager"
    ])
    while os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
        time.sleep(3)
    print("✅ vLLM Server đã sẵn sàng!
")
else:
    print("⚡ vLLM Server đang hoạt động sẵn sàng!
")

# 6. Chạy Pipeline
print("🚀 [1/2] Bắt đầu xử lý: BENCHMARK 300 CÂU MẪU...")
!python3 run_pipeline.py \
    --questions train/sample_300_questions.jsonl \
    --evaluate \
    --gold train/sample_300_answers.jsonl \
    --data-root train/data \
    --calendar train/calendar.csv \
    --rulebook train/rulebook.md \
    --output predictions_sample300.jsonl \
    --backend openai \
    --api-base http://localhost:8000/v1 \
    --concurrency $CONCURRENCY \
    --fresh

# 7. Lưu file kết quả về Google Drive
print("\n💾 [2/2] Đang lưu file predictions_sample300.jsonl về Google Drive...")
local_pred = Path("predictions_sample300.jsonl")
drive_pred = Path("/content/drive/MyDrive/predictions_sample300.jsonl")
if local_pred.exists():
    shutil.copy(str(local_pred), str(drive_pred))
    print(f"✅ Đã lưu kết quả tại: {drive_pred}")

if AUTO_SHUTDOWN:
    print("\n🎉 Hoàn tất! Tự động ngắt GPU để tiết kiệm Compute Units...")
    from google.colab import runtime
    runtime.unassign()


## 🟡 Cell 2: Chạy Benchmark toàn bộ 3.000 câu tập Train (Train Full — ~35 đến 45 phút)

In [ ]:
# ==============================================================
# CELL 2: BENCHMARK TOÀN BỘ 3.000 CÂU TRAIN
# Benchmark toàn bộ 3.000 câu tập Train (Thời gian ~35 đến 45 phút)
# ==============================================================
from google.colab import drive
import os, zipfile, subprocess, time, shutil
from pathlib import Path

# CẤU HÌNH
AUTO_SHUTDOWN = True  # True = tự động tắt GPU sau khi xong để tiết kiệm Compute Units
CONCURRENCY = 8        # Số luồng chạy song song trên GPU

# 1. Mount Google Drive
drive.mount('/content/drive')
WORK_DIR = Path('/content/Datamate')

# 2. TỰ ĐỘNG KÉO CODE MỚI NHẤT TỪ GITHUB (Mất 0.5s)
if not WORK_DIR.exists():
    print("🚀 Đang tải code từ GitHub...")
    !git clone https://github.com/25520969-netizen/Datamate.git $WORK_DIR
    %cd $WORK_DIR
else:
    %cd $WORK_DIR
    print("⚡ Đang cập nhật code mới nhất từ GitHub...")
    !git pull

# 3. Nạp Data từ Drive vào SSD Colab (Chỉ giải nén LẦN ĐẦU TIÊN)
DRIVE_ZIP = Path('/content/drive/MyDrive/zalo_data.zip')
if not (WORK_DIR / 'train' / 'data').exists():
    if not DRIVE_ZIP.exists():
        raise FileNotFoundError("Chưa tìm thấy 'MyDrive/zalo_data.zip'. Vui lòng tải file lên Google Drive trước!")
    print("📦 Lần đầu chạy: Đang giải nén bộ dữ liệu 183 MB vào SSD Colab...")
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(WORK_DIR)
    print("✅ Đã nạp xong toàn bộ dữ liệu Train & Public Test!")
else:
    print("⚡ Dữ liệu đã có sẵn trên SSD Colab, không cần giải nén lại!")

# 4. Cài đặt thư viện cần thiết (nếu thiếu)
!pip install -q vllm duckdb openpyxl pyarrow pytest

# 5. Khởi động vLLM Server (nếu chưa chạy)
if os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
    print("⏳ Đang khởi động vLLM Server trên GPU Colab (chế độ siêu tốc ~45s)...")
    os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")
    subprocess.Popen([
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
        "--port", "8000",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--enable-prefix-caching",
        "--enforce-eager"
    ])
    while os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
        time.sleep(3)
    print("✅ vLLM Server đã sẵn sàng!
")
else:
    print("⚡ vLLM Server đang hoạt động sẵn sàng!
")

# 6. Chạy Pipeline
print("🚀 [1/2] Bắt đầu xử lý: BENCHMARK TOÀN BỘ 3.000 CÂU TRAIN...")
!python3 run_pipeline.py \
    --questions train/questions.jsonl \
    --evaluate \
    --gold train/answers.jsonl \
    --data-root train/data \
    --calendar train/calendar.csv \
    --rulebook train/rulebook.md \
    --output predictions_train3000.jsonl \
    --backend openai \
    --api-base http://localhost:8000/v1 \
    --concurrency $CONCURRENCY \
    --fresh

# 7. Lưu file kết quả về Google Drive
print("\n💾 [2/2] Đang lưu file predictions_train3000.jsonl về Google Drive...")
local_pred = Path("predictions_train3000.jsonl")
drive_pred = Path("/content/drive/MyDrive/predictions_train3000.jsonl")
if local_pred.exists():
    shutil.copy(str(local_pred), str(drive_pred))
    print(f"✅ Đã lưu kết quả tại: {drive_pred}")

if AUTO_SHUTDOWN:
    print("\n🎉 Hoàn tất! Tự động ngắt GPU để tiết kiệm Compute Units...")
    from google.colab import runtime
    runtime.unassign()


## 🔴 Cell 3: Chạy 1.000 câu thi Public Test (Xuất submission.jsonl — ~12 đến 15 phút)

In [ ]:
# ==============================================================
# CELL 3: 1.000 CÂU THI PUBLIC TEST NỘP BÀI
# Chạy 1.000 câu thi Public Test nộp Zalo AI (Thời gian ~12 đến 15 phút)
# ==============================================================
from google.colab import drive
import os, zipfile, subprocess, time, shutil
from pathlib import Path

# CẤU HÌNH
AUTO_SHUTDOWN = True  # True = tự động tắt GPU sau khi xong để tiết kiệm Compute Units
CONCURRENCY = 8        # Số luồng chạy song song trên GPU

# 1. Mount Google Drive
drive.mount('/content/drive')
WORK_DIR = Path('/content/Datamate')

# 2. TỰ ĐỘNG KÉO CODE MỚI NHẤT TỪ GITHUB (Mất 0.5s)
if not WORK_DIR.exists():
    print("🚀 Đang tải code từ GitHub...")
    !git clone https://github.com/25520969-netizen/Datamate.git $WORK_DIR
    %cd $WORK_DIR
else:
    %cd $WORK_DIR
    print("⚡ Đang cập nhật code mới nhất từ GitHub...")
    !git pull

# 3. Nạp Data từ Drive vào SSD Colab (Chỉ giải nén LẦN ĐẦU TIÊN)
DRIVE_ZIP = Path('/content/drive/MyDrive/zalo_data.zip')
if not (WORK_DIR / 'public_test' / 'data').exists():
    if not DRIVE_ZIP.exists():
        raise FileNotFoundError("Chưa tìm thấy 'MyDrive/zalo_data.zip'. Vui lòng tải file lên Google Drive trước!")
    print("📦 Lần đầu chạy: Đang giải nén bộ dữ liệu 183 MB vào SSD Colab...")
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(WORK_DIR)
    print("✅ Đã nạp xong toàn bộ dữ liệu Train & Public Test!")
else:
    print("⚡ Dữ liệu đã có sẵn trên SSD Colab, không cần giải nén lại!")

# 4. Cài đặt thư viện cần thiết (nếu thiếu)
!pip install -q vllm duckdb openpyxl pyarrow pytest

# 5. Khởi động vLLM Server (nếu chưa chạy)
if os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
    print("⏳ Đang khởi động vLLM Server trên GPU Colab (chế độ siêu tốc ~45s)...")
    os.system("fuser -k 8000/tcp 2>/dev/null || true; pkill -9 -f vllm 2>/dev/null || true")
    subprocess.Popen([
        "python3", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-Coder-7B-Instruct",
        "--port", "8000",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.90",
        "--enable-prefix-caching",
        "--enforce-eager"
    ])
    while os.system("curl -s http://localhost:8000/v1/models > /dev/null") != 0:
        time.sleep(3)
    print("✅ vLLM Server đã sẵn sàng!
")
else:
    print("⚡ vLLM Server đang hoạt động sẵn sàng!
")

# 6. Chạy Pipeline
print("🚀 [1/2] Bắt đầu xử lý: 1.000 CÂU THI PUBLIC TEST NỘP BÀI...")
!python3 run_pipeline.py \
    --questions public_test/questions.jsonl \
    --data-root public_test/data \
    --calendar public_test/calendar.csv \
    --rulebook public_test/rulebook.md \
    --output submission.jsonl \
    --backend openai \
    --api-base http://localhost:8000/v1 \
    --concurrency $CONCURRENCY \
    --fresh

# 7. Lưu file kết quả về Google Drive
print("\n💾 [2/2] Đang lưu file submission.jsonl về Google Drive...")
local_pred = Path("submission.jsonl")
drive_pred = Path("/content/drive/MyDrive/submission.jsonl")
if local_pred.exists():
    shutil.copy(str(local_pred), str(drive_pred))
    print(f"✅ Đã lưu kết quả tại: {drive_pred}")

if AUTO_SHUTDOWN:
    print("\n🎉 Hoàn tất! Tự động ngắt GPU để tiết kiệm Compute Units...")
    from google.colab import runtime
    runtime.unassign()
